# 한국어 데이터로 에이전트 테스트하기

이 노트북은 한국어 공공데이터를 이용해 multilingual retrieval과 한국어 답변 품질을 실험한다. 특히 현재 전처리와 토큰화 로직이 영문 중심으로 설계돼 있기 때문에, multilingual embedding을 쓰더라도 한국어에서 어떤 한계가 드러나는지 직접 확인하는 것이 중요하다.

## 학습 목표
- 한국어 retrieval이 영문 retrieval과 다른 이유를 토큰화와 청킹 관점에서 설명할 수 있다.
- `sentence_split()`, `content_tokens()`, `STOPWORDS`가 한국어에서 어떤 한계를 가지는지 이해한다.
- 한국어 질문에 대한 workflow 품질과 grounding을 영문 데이터와 비교해 읽을 수 있다.
- 형태소 분석기, 한국어 stopword, 한국어 특화 임베딩 같은 개선 아이디어를 구체적으로 말할 수 있다.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RuntimeConfig
from src.data_profiles import load_profile

runtime_config = RuntimeConfig.auto_detect()
korean_profile = load_profile('korean_public', persist=False)
english_profile = load_profile('demo', persist=False)

print(sys.executable)
{
    'device': runtime_config.device,
    'embedding_model': runtime_config.embedding_model,
    'llm_available': runtime_config.llm_available,
    'korean_profile_stats': korean_profile['stats'],
}

## 구현: 한국어 텍스트 처리의 특수성

**목적**
        - 현재 `utils.py`의 토큰화와 문장 분리 로직이 한국어에 얼마나 적합한지 코드 수준에서 점검한다.

        **핵심 로직**
        - `STOPWORDS`는 영문 불용어 중심이라 한국어 조사/어미를 거의 제거하지 못한다.
        - `content_tokens()`는 정규식 기반 영문/숫자 토큰 추출에 최적화돼 있어 한국어 어절과 형태소를 세밀하게 다루지 못한다.
        - `sentence_split()`은 문장 부호와 빈 줄 기준 분할이라 한국어에도 어느 정도 동작하지만, 문장 종결 표현이 다양할수록 한계가 생긴다.

        **실제 소스 코드: STOPWORDS — src/utils.py**
        ```python
        STOPWORDS = {
    "a",
    "an",
    "answer",
    "and",
    "are",
    "as",
    "at",
    "be",
    "by",
    "for",
    "from",
    "grounded",
    "how",
    "in",
    "into",
    "is",
    "it",
    "source",
    "sources",
    "summary",
    "comparison",
    "of",
    "on",
    "or",
    "that",
    "the",
    "their",
    "this",
    "to",
    "was",
    "what",
    "when",
    "which",
    "who",
    "why",
    "with",
}
        ```

        **실제 소스 코드: content_tokens() — src/utils.py**
        ```python
        def content_tokens(text: str) -> list[str]:
    normalized = [normalize_token(token) for token in tokenize(text)]
    return [token for token in normalized if token not in STOPWORDS]
        ```

        **실제 소스 코드: sentence_split() — src/utils.py**
        ```python
        def sentence_split(text: str) -> list[str]:
    cleaned = text.replace("\r", "\n")
    rough_parts = re.split(r"(?<=[.!?])\s+|\n{2,}", cleaned)
    return [normalize_text(part) for part in rough_parts if normalize_text(part)]
        ```

        **코드 읽기 포인트**
        - `WORD_RE = re.compile(r"[a-z0-9]+", re.IGNORECASE)`가 사실상 영문/숫자 토큰만 적극적으로 뽑고 있다는 점이 핵심 한계다.
        - `STOPWORDS`에 `은/는/이/가/을/를` 같은 한국어 불용어가 없어, 한국어 retrieval에서는 noise token 비중이 높아질 수 있다.
        - `sentence_split()`은 문장 종결 부호 중심이라 baseline 수준에서는 쓸 만하지만, 한국어 구어체/정책문서 표현까지 잘 분리한다고 보긴 어렵다.

        **결과 해석 가이드**
        - 한국어 `content_tokens_preview`가 기대보다 빈약하거나 어색하면 현재 tokenizer가 한국어 정보를 충분히 살리지 못하고 있다는 뜻이다.
        - `chunk_preview`가 지나치게 길거나 문장 경계가 어색하면 chunking도 한국어 최적화가 필요하다는 신호다.

        **💡 면접 포인트**
        - multilingual 모델을 쓴다고 해서 전처리까지 자동으로 한국어 친화적으로 바뀌는 것은 아니다.
        - 한국어 retrieval 품질은 임베딩뿐 아니라 형태소 분석, 불용어 처리, 문장 분할 품질의 영향을 크게 받는다.


In [ ]:
import pandas as pd

from src.ingestion import chunk_document
from src.utils import content_tokens, sentence_split

korean_doc = korean_profile['documents'][0]
english_doc = english_profile['documents'][0]

korean_sentences = sentence_split(korean_doc['text'])[:5]
english_sentences = sentence_split(english_doc['text'])[:5]

korean_chunks = chunk_document(korean_doc)[:3]
english_chunks = chunk_document(english_doc)[:3]

comparison_rows = [
    {
        'language': 'ko',
        'source': korean_doc['source'],
        'sentence_count_preview': len(korean_sentences),
        'content_tokens_preview': content_tokens(korean_doc['text'])[:12],
        'chunk_preview': korean_chunks[0]['text'][:140] if korean_chunks else '',
    },
    {
        'language': 'en',
        'source': english_doc['source'],
        'sentence_count_preview': len(english_sentences),
        'content_tokens_preview': content_tokens(english_doc['text'])[:12],
        'chunk_preview': english_chunks[0]['text'][:140] if english_chunks else '',
    },
]

print('Korean sentence preview:', korean_sentences)
print('English sentence preview:', english_sentences)
pd.DataFrame(comparison_rows)

## 실험: 한국어 vs 영문 검색 품질 비교

**목적**
- 같은 backend 설정에서 한국어와 영문 corpus의 retrieval score 분포를 비교한다.

**핵심 로직**
- `backend='faiss'`를 요청하되, 환경에 따라 실제 backend가 TF-IDF로 폴백될 수도 있다.
- 한국어 public data와 demo 영문 corpus에서 같은 query type 하나를 골라 top-k score 분포를 비교한다.

**주요 파라미터**
- `requested_backend`: 실험에서 의도한 retrieval backend다.
- `actual_backend_ko`, `actual_backend_en`: 실제로 어떤 retriever가 사용됐는지 알려 준다.
- `embedding_model_setting`: 런타임 설정상 어떤 임베딩 모델을 기대하는지 보여 준다.

**결과 해석 가이드**
- 한국어 score가 영문보다 낮다고 해서 multilingual retrieval이 실패라고 단정하면 안 된다. corpus 구조와 토큰화 한계가 함께 작용할 수 있다.
- `actual_backend_*`가 `HybridRetriever`라면 이번 실험은 dense retrieval이 아니라 TF-IDF 기반 비교일 수 있다.
- 상위 source가 질문 의도와 맞는지 보는 것이 단순 score 숫자보다 더 중요하다.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.data_profiles import load_profile

requested_backend = 'faiss'
korean_retrieval_profile = load_profile('korean_public', backend=requested_backend, persist=False)
english_retrieval_profile = load_profile('demo', backend=requested_backend, persist=False)

actual_backend_ko = type(korean_retrieval_profile['retriever']).__name__
actual_backend_en = type(english_retrieval_profile['retriever']).__name__

korean_eval_frame = pd.DataFrame(korean_retrieval_profile['eval_dataset']).sort_values(['question_type', 'id'])
english_eval_frame = pd.DataFrame(english_retrieval_profile['eval_dataset']).sort_values(['question_type', 'id'])

selected_query_type = 'simple_lookup'
korean_query = korean_eval_frame[korean_eval_frame['question_type'] == selected_query_type].iloc[0]['question']
english_query = english_eval_frame[english_eval_frame['question_type'] == selected_query_type].iloc[0]['question']

korean_hits = korean_retrieval_profile['retriever'].search(korean_query, top_k=5)
english_hits = english_retrieval_profile['retriever'].search(english_query, top_k=5)

score_frame = pd.DataFrame(
    [
        {'language': 'ko', 'rank': index + 1, 'score': item['score'], 'source': item['source'], 'query': korean_query}
        for index, item in enumerate(korean_hits)
    ] + [
        {'language': 'en', 'rank': index + 1, 'score': item['score'], 'source': item['source'], 'query': english_query}
        for index, item in enumerate(english_hits)
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
for language, frame in score_frame.groupby('language'):
    ax.plot(frame['rank'], frame['score'], marker='o', label=language)
ax.set_title(f'Score distribution by language (requested backend={requested_backend})')
ax.set_xlabel('rank')
ax.set_ylabel('score')
ax.legend()
plt.tight_layout()
plt.show()

{
    'requested_backend': requested_backend,
    'actual_backend_ko': actual_backend_ko,
    'actual_backend_en': actual_backend_en,
    'embedding_model_setting': runtime_config.embedding_model,
}
score_frame[['language', 'rank', 'score', 'source']]

## 실험: 한국어 LLM 답변 품질

**목적**
- Qwen 계열 LLM이 한국어 질문에 대해 grounded answer를 얼마나 자연스럽게 내는지 본다.

**핵심 로직**
- 한국어 eval dataset에서 insufficent-evidence가 아닌 질문 하나를 골라 `run_workflow(use_llm=True)`를 실행한다.
- live/fallback/server_unavailable 상태를 함께 기록해, 실제 LLM 경로가 탔는지도 구분한다.

**주요 파라미터**
- `korean_question`: 실험용 한국어 질문
- `llm_path`: live/fallback/server_unavailable 구분값
- `coverage_score`: verifier 기준 한국어 답변의 grounded 정도

**결과 해석 가이드**
- 한국어 답변이 자연스러워도 `coverage_score`가 낮으면 근거 보수성이 떨어질 수 있다.
- `query_type`이 예상과 크게 다르면 classifier가 한국어 표현을 충분히 반영하지 못했을 가능성이 있다.
- fallback이면 이번 셀은 LLM 품질보다 폴백 안전성 확인으로 읽는 편이 맞다.

**💡 면접 포인트**
- multilingual LLM이 한국어를 꽤 잘 생성하더라도, retrieval과 verifier가 영문 중심이면 전체 시스템 품질은 제한될 수 있다.
- 한국어 생성 품질과 한국어 grounding 품질은 별개로 봐야 한다.


In [ ]:
import warnings

from src.llm_client import OllamaClient
from src.workflow import run_workflow

korean_client = OllamaClient()
korean_llm_live = korean_client.is_available()
korean_question = pd.DataFrame(korean_profile['eval_dataset']).query("question_type != 'insufficient_evidence_risk'").iloc[0]['question']

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    korean_state = run_workflow(
        korean_question,
        retriever=korean_profile['retriever'],
        use_llm=True,
        llm_client=korean_client,
    )

{
    'question': korean_question,
    'llm_path': 'live' if korean_llm_live and not caught else ('fallback' if caught else 'server_unavailable'),
    'query_type': korean_state['query_type'],
    'final_status': korean_state['final_status'],
    'coverage_score': korean_state['verification_result'].coverage_score,
    'final_answer': korean_state['final_answer'],
}

## 결과 해석: 평가

**목적**
- 한국어 전체 eval set과 영문 demo subset을 같은 방식으로 평가해 평균 차이를 본다.

**핵심 로직**
- `answer_correctness`, `retrieval_hit_rate`, `grounding_pass_rate`, `latency_seconds`를 profile별로 평균낸다.

**결과 해석 가이드**
- `answer_correctness`와 `retrieval_hit_rate`가 같이 낮으면 retrieval과 synthesis가 모두 어려운 상황일 수 있다.
- retrieval은 괜찮은데 grounding만 낮다면, 한국어 답변이 더 자연스럽게 확장되며 unsupported claim이 늘었을 수 있다.
- latency 차이는 corpus 길이뿐 아니라 LLM의 한국어 생성 길이 차이에서도 올 수 있다.


In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluator import retrieval_hit_rate, score_answer
from src.workflow import run_workflow


def sample_balanced(dataset: list[dict], count_per_type: int) -> list[dict]:
    frame = pd.DataFrame(dataset).sort_values(['question_type', 'id'])
    return (
        frame.groupby('question_type', as_index=False, group_keys=False)
        .head(count_per_type)
        .to_dict(orient='records')
    )


def evaluate_profile(profile_name: str, profile: dict, samples: list[dict]) -> pd.DataFrame:
    rows = []
    for sample in samples:
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            state = run_workflow(
                sample['question'],
                retriever=profile['retriever'],
                use_llm=True,
                llm_client=korean_client,
            )
        latency = round(time.perf_counter() - start, 4)
        rows.append(
            {
                'profile': profile_name,
                'question_id': sample['id'],
                'question_type': sample['question_type'],
                'answer_correctness': score_answer(
                    state['final_answer'],
                    sample['gold_answer'],
                    state['final_status'],
                    sample.get('expected_status', 'answered'),
                ),
                'retrieval_hit_rate': retrieval_hit_rate(
                    state['retrieved_docs'],
                    sample.get('expected_sources', []),
                ),
                'grounding_pass_rate': float(state['verification_result'].is_grounded),
                'latency_seconds': latency,
                'path_mode': 'live' if korean_llm_live and not caught else ('fallback' if caught else 'server_unavailable'),
            }
        )
    return pd.DataFrame(rows)

korean_samples = korean_profile['eval_dataset']
english_samples = sample_balanced(english_profile['eval_dataset'], count_per_type=4)

korean_eval = evaluate_profile('korean_public', korean_profile, korean_samples)
english_eval = evaluate_profile('demo_en', english_profile, english_samples)
combined_eval = pd.concat([korean_eval, english_eval], ignore_index=True)
summary = (
    combined_eval.groupby('profile', as_index=False)
    .agg(
        answer_correctness=('answer_correctness', 'mean'),
        retrieval_hit_rate=('retrieval_hit_rate', 'mean'),
        grounding_pass_rate=('grounding_pass_rate', 'mean'),
        latency_seconds=('latency_seconds', 'mean'),
    )
    .round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.plot(x='profile', y=['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate'], kind='bar', ax=axes[0], color=['#4C78A8', '#54A24B', '#E45756'])
axes[0].set_ylim(0, 1)
axes[0].set_title('Quality comparison')
axes[0].set_ylabel('score')
summary.plot(x='profile', y='latency_seconds', kind='bar', ax=axes[1], color='#F58518', legend=False)
axes[1].set_title('Average latency')
axes[1].set_ylabel('seconds')
plt.tight_layout()
plt.show()

summary

## 한국어 특화 이슈 + 개선 아이디어

현재 구현은 교육용 baseline이라 한국어에 최적화돼 있지 않다. 그래서 개선 방향도 비교적 명확하다.

- 형태소 분석기(mecab, konlpy)를 붙이면 조사/어미와 핵심 명사를 더 안정적으로 분리할 수 있다.
- 한국어 불용어 리스트를 추가하면 retrieval noise를 줄일 수 있다.
- `ko-sroberta`, `bge-m3`, 한국어 특화 sentence embedding 모델과의 비교 실험도 가능하다.
- 정책/법령 문서는 문장 구조가 길고 조항 번호가 많으므로, chunking을 조항 기준이나 문단 기준으로 바꾸는 것도 효과적일 수 있다.

**💡 면접 포인트**
- multilingual 모델의 한국어 한계를 실험으로 확인했고, 토큰화·불용어·임베딩·chunking 네 축에서 개선안을 제시할 수 있다.
- 한국어 품질 문제를 단순히 "모델이 약하다"로 보지 않고, retrieval stack 전체 문제로 설명할 수 있다.


## 핵심 정리

이 노트북을 통해 한국어 데이터 실험은 단순 번역판 영문 RAG가 아니라, 별도 고려가 필요한 retrieval 문제라는 점을 확인했다. 현재 `sentence_split()`, `content_tokens()`, `STOPWORDS`는 영문 중심이라 한국어의 조사·어미·형태소 구조를 충분히 반영하지 못한다. 그 결과 multilingual embedding을 써도 retrieval과 grounding 품질이 제한될 수 있다.

**💡 면접 포인트**
- multilingual 모델의 한국어 한계를 실험으로 확인했고, 형태소 분석 + 한국어 불용어 + 한국어 특화 임베딩 비교로 개선 가능하다고 설명할 수 있다.
- 한국어 retrieval 품질은 모델 하나가 아니라 tokenizer, chunking, stopword, verifier까지 포함한 전체 스택 문제로 봐야 한다.
- 영문에서 잘 되던 시스템을 한국어에 적용할 때는 preprocessing 계층을 먼저 의심하는 것이 효율적이다.
